In [1]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

## Data Enrichment with AI

### Setup

##### Create a dataset for storing the AI tables and another for storing the AI models

In [ ]:
%%bigquery
CREATE SCHEMA `auspicious-lead-382321`.movie_stg_ai

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE SCHEMA `auspicious-lead-382321`.remote_models

Executing query with job ID: ce31eaa5-08d9-4b63-95cd-ac83e69a42de
Query executing: 0.98s


ERROR:
 409 Already Exists: Dataset auspicious-lead-382321:remote_models; reason: duplicate, message: Already Exists: Dataset auspicious-lead-382321:remote_models

Location: US
Job ID: ce31eaa5-08d9-4b63-95cd-ac83e69a42de



##### Before running this cell, create the remote connection and assign the IAM role `Vertex AI User` to the service account associated with the connection.

In [4]:
%%bigquery
create or replace model remote_models.gemini_pro
  remote with connection `projects/auspicious-lead-382321/locations/us/connections/vertex_connection`
  options (endpoint = 'gemini-pro');

Query is running:   0%|          |

""


### Part 1: Predict the category of a snack

In [ ]:
%%bigquery
select * except (data_source, load_time)
from movie_stg.Snack
where name is not null
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein
0,79,Banana Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,99.0,NaN
1,75,Pretzel Bites: Add Salt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN
2,78,Banana Smoothie 16 fl oz,260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,66.0,NaN
3,59,Apple Sauce,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,75.0,10.0,NaN,8.0,NaN
4,64,Pickle,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2790.0,NaN,6.0,NaN,NaN,NaN


#### Test the generate_text function

In [5]:
%%bigquery
declare prompt_query STRING default "Suggest a category for each snack. Return the output as json, include the id in the output";
select *
from ML.generate_text(
  model remote_models.gemini_pro,
  (
    select concat(prompt_query, to_json_string(json_object("id", id, "name", name))) as prompt
    from movie_stg.Snack
    order by id
    limit 10
  ),
  struct(TRUE as flatten_json_output)
);

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,ml_generate_text_rai_result,ml_generate_text_status,prompt
0,"```json\n[\n {\n ""id"": 0,\n ""name"": ""Ho...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
1,"```json\n[\n {\n ""id"": 1,\n ""name"": ""Ho...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
2,"```json\n{""id"":2,""name"":""House-made Caramel Co...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
3,"```json\n[\n {\n ""id"": 3,\n ""name"": ""Ho...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
4,"```json\n[\n {\n ""id"": 4,\n ""name"": ""Ho...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
5,"```json\n[\n {""id"":5,""name"":""House-made Chees...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
6,"```json\n{""id"":6,""name"":""Plain Popcorn (No But...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
7,"```json\n[\n {\n ""id"": 7,\n ""name"": ""Pl...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
8,"```json\n{""id"":8,""name"":""Plain Popcorn Refilla...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...
9,"```json\n[\n {\n ""id"": 9,\n ""name"": ""Pl...","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a category for each snack. Return the ...


#### Tweak the prompt and save the output
##### [More details](https://cloud.google.com/bigquery/docs/generate-text#generate_text_from_text_data_by_using_a_prompt_from_a_query) on `ML.generate_text` parameters

In [ ]:
%%bigquery
declare prompt_query STRING default "Suggest a category for each snack. Return the output as json, include only the id and category in the output";
create or replace table movie_stg_ai.category_predictions_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name))) as prompt
      from movie_stg.Snack
      order by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n{""id"":2,""category"":""Popcorn""}\n```"
1,"```json\n{""id"":6,""category"":""Popcorn""}\n```"
2,"```json\n{""id"":21,""category"":""Candy""}\n```"
3,"```json\n{""id"":4,""category"":""Sides""}\n```"
4,"```json\n{""id"":14,""category"":""Sides""}\n```"
...,...
95,"```json\n{""id"":60,""category"":""Sauce""}\n```"
96,"```json\n{""id"":25,""category"":""Chips""}\n```"
97,"```json\n{""id"":32,""category"":""Burgers""}\n```"
98,"```json\n{""id"":45,""category"":""Hot Dog""}\n```"


#### Format the output to proper json

In [ ]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n{""id"":2,""category"":""Popcorn""}\n```","{""id"":2,""category"":""Popcorn""}"
1,"```json\n{""id"":6,""category"":""Popcorn""}\n```","{""id"":6,""category"":""Popcorn""}"
2,"```json\n{""id"":21,""category"":""Candy""}\n```","{""id"":21,""category"":""Candy""}"
3,"```json\n{""id"":4,""category"":""Sides""}\n```","{""id"":4,""category"":""Sides""}"
4,"```json\n{""id"":14,""category"":""Sides""}\n```","{""id"":14,""category"":""Sides""}"
...,...,...
95,"```json\n{""id"":60,""category"":""Sauce""}\n```","{""id"":60,""category"":""Sauce""}"
96,"```json\n{""id"":25,""category"":""Chips""}\n```","{""id"":25,""category"":""Chips""}"
97,"```json\n{""id"":32,""category"":""Burgers""}\n```","{""id"":32,""category"":""Burgers""}"
98,"```json\n{""id"":45,""category"":""Hot Dog""}\n```","{""id"":45,""category"":""Hot Dog""}"


In [ ]:
%%bigquery
create or replace table movie_stg_ai.category_predictions_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.category') as category
from movie_stg_ai.category_predictions_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,category
0,14,Sides
1,52,Sandwiches
2,39,Canned Foods
3,55,Beverages
4,10,Popcorn
...,...,...
95,11,Popcorn
96,64,Condiments
97,41,Sides
98,38,Fried


#### Add the subcategory field to the Snack table

In [ ]:
%%bigquery
alter table movie_stg.Snack add column category string;

Query is running:   0%|          |

""


#### Update the Snack records with the predicted subcategory

In [ ]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id')
  from movie_stg_ai.category_predictions_formatted
    where 1=1


Query is running:   0%|          |

Downloading:   0%|          |

,f0_
0,14
1,52
2,39
3,55
4,10
...,...
95,11
96,64
97,41
98,38


In [ ]:
%%bigquery
update movie_stg.Snack set category =
  (select json_value(ml_generate_text_llm_result, '$.category')
   from movie_stg_ai.category_predictions_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


#### Inspect the output

In [ ]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where category is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,88,Strawberry Smoothie 16 fl oz,260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN,Smoothies,None
1,75,Pretzel Bites: Add Salt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,Salty Snacks,None
2,101,Gummy Bears,200.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Candy,None
3,55,Gold Peak Brewed Tea - 30 fl oz,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,330.0,3.0,NaN,NaN,NaN,Beverages,None
4,64,Pickle,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2790.0,NaN,6.0,NaN,NaN,NaN,Condiments,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,Vanilla Shake,690.0,340.0,38.0,25.0,1.0,0.0,1.0,135.0,270.0,25.0,74.0,NaN,64.0,10.0,Shakes,None
96,94,Strawberry Shake,770.0,340.0,38.0,25.0,1.0,0.0,1.0,135.0,270.0,25.0,93.0,NaN,83.0,10.0,Drinks,None
97,91,Chocolate Shake,830.0,340.0,38.0,25.0,1.0,0.0,1.0,135.0,270.0,25.0,107.0,3.0,89.0,12.0,Drinks,None
98,93,Espresso Shake,690.0,340.0,38.0,25.0,1.0,0.0,1.0,135.0,280.0,90.0,74.0,NaN,64.0,11.0,Coffee,None


In [59]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where category is not null

Query is running:   0%|          |

""


### Detect whether a snack is healthy or unhealthy

In [7]:
%%bigquery
declare prompt_query STRING default "Detect whether a snack is healthy or unhealthy. A snack is unhealthy if it has high calories or high sugar or high calories_from_fat or hight total_fat or high sodium, or a combination of these. Include the id in the output as well";
create or replace table movie_stg_ai.is_healthy_predictions_raw as
select *
from ML.generate_text(
  model remote_models.gemini_pro,
  (
    select concat(prompt_query, to_json_string(json_object("id", id, "name", name,
                   "category", category, "calories", calories, "calories_from_fat", calories_from_fat, "total_fat", total_fat, "sodium", sodium))) as prompt
    from movie_stg.Snack
    order by id
  ),
  struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
);

Executing query with job ID: b8b082cf-026a-45f9-8d58-211cfe934803
Query executing: 0.63s


ERROR:
 400 Not found: Dataset auspicious-lead-382321:movie_stg_ai was not found in location US at [2:1]

Location: US
Job ID: b8b082cf-026a-45f9-8d58-211cfe934803



In [ ]:
%%bigquery
select ml_generate_text_llm_result, prompt
from movie_stg_ai.is_healthy_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,prompt
0,"{""calories"":390.0,""calories_from_fat"":null,""ca...",Detect whether a snack is healthy or unhealthy...
1,"{""calories"":1450.0,""calories_from_fat"":810.0,""...",Detect whether a snack is healthy or unhealthy...
2,"{""calories"":480.0,""calories_from_fat"":190.0,""c...",Detect whether a snack is healthy or unhealthy...
3,"{""id"": 7, ""is_healthy"": false}",Detect whether a snack is healthy or unhealthy...
4,"{""calories"":1360.0,""calories_from_fat"":790.0,""...",Detect whether a snack is healthy or unhealthy...
...,...,...
95,"{""calories"":740.0,""calories_from_fat"":250.0,""c...",Detect whether a snack is healthy or unhealthy...
96,"{""calories"":0.0,""calories_from_fat"":null,""cate...",Detect whether a snack is healthy or unhealthy...
97,The snack with id 93 is unhealthy because it h...,Detect whether a snack is healthy or unhealthy...
98,Healthy,Detect whether a snack is healthy or unhealthy...


In [ ]:
%%bigquery
create or replace table movie_stg_ai.is_healthy_predictions_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.is_healthy_predictions_raw

Query is running:   0%|          |

""


In [6]:
%%bigquery
select * from movie_stg_ai.is_healthy_predictions_formatted

Executing query with job ID: 183442ce-f2ee-4f3a-a5f7-9b29b69d8be5
Query executing: 0.77s


ERROR:
 404 Not found: Table auspicious-lead-382321:movie_stg_ai.is_healthy_predictions_formatted was not found in location us-central1; reason: notFound, message: Not found: Table auspicious-lead-382321:movie_stg_ai.is_healthy_predictions_formatted was not found in location us-central1

Location: us-central1
Job ID: 183442ce-f2ee-4f3a-a5f7-9b29b69d8be5



In [ ]:
%%bigquery
alter table movie_stg.Snack add column is_healthy string;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
update movie_stg.Snack set is_healthy =
  (select json_value(ml_generate_text_llm_result, '$.is_healthy')
   from movie_stg_ai.is_healthy_predictions_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where is_healthy is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,82,Mango Smoothie 24 fl oz,400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.0,NaN,108.0,NaN,103.0,NaN,Smoothies,false
1,86,Strawberry Banana Smoothie 16 fl oz,260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN,Smoothies,false
2,78,Banana Smoothie 16 fl oz,260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,66.0,NaN,Smoothies,false
3,72,Pretzel Bites: Add Honey Mustard,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,Salty Snacks,true
4,79,Banana Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,99.0,NaN,Smoothies,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,17,Plain Popcorn (No Butter) Medium (coconut oil),600.0,230.0,27.0,19.0,NaN,2.5,2.5,NaN,1530.0,350.0,82.0,15.0,NaN,14.0,Popcorn,false
89,41,Curly Fries (Fried),630.0,390.0,44.0,9.0,0.0,13.0,5.0,NaN,1130.0,700.0,54.0,5.0,3.0,5.0,Sides,false
90,25,Cheddar Crunch 85 fl oz,4550.0,3010.0,341.0,196.0,NaN,22.0,14.0,105.0,7010.0,1530.0,306.0,5.0,26.0,52.0,Chips,false
91,21,Caramel Corn 85 fl oz,2340.0,560.0,64.0,43.0,NaN,NaN,NaN,NaN,2980.0,830.0,446.0,NaN,276.0,21.0,Candy,false


In [61]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where is_healthy is not null

Query is running:   0%|          |

""


### Part 3: Detect the service category of a snack in snack service

#### Explore the raw data

In [15]:
%%bigquery
create or replace model remote_models.gemini_pro
  remote with connection `projects/auspicious-lead-382321/locations/us/connections/vertex_connection`
  options (endpoint = 'gemini-pro');

Query is running:   0%|          |

""


In [3]:
%%bigquery
select snack_id, service_category, category
from movie_stg.Snack_Service
where service_category is null
and category is not null
order by snack_id

Query is running:   0%|          |

Downloading:   0%|          |

,snack_id,service_category,category
0,0,None,Popcorn
1,0,None,Popcorn
2,0,None,Popcorn
3,1,None,Popcorn
4,1,None,Popcorn
...,...,...,...
3224,99,None,Desserts
3225,99,None,Desserts
3226,99,None,Desserts
3227,99,None,Desserts


In [5]:
%%bigquery
select * from movie_stg.Snack_Service ss
LEFT JOIN movie_stg.Snack s ON  ss.snack_id = s.id


Query is running:   0%|          |

Downloading:   0%|          |

,theater_name,snack_id,theater_id,theater_chain_name,service_category,data_source,load_time,category,id,name,...,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source_1,load_time_1,category_1,is_healthy
0,Atlas Cinema Great Lakes Stadium 16,61,4424,Atlas Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Dips,61,Buffalo Dipping Sauce,...,1650.0,NaN,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Dips,false
1,Atlas Cinemas Midway Mall 8,61,4626,Atlas Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Dips,61,Buffalo Dipping Sauce,...,1650.0,NaN,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Dips,false
2,Atlas Lakeshore 7,61,838,Atlas Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Dips,61,Buffalo Dipping Sauce,...,1650.0,NaN,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Dips,false
3,Atlas Cinemas Eastgate 10,61,4850,Atlas Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Dips,61,Buffalo Dipping Sauce,...,1650.0,NaN,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Dips,false
4,Atlas Cinemas Diamond Ctr 16,61,840,Atlas Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Dips,61,Buffalo Dipping Sauce,...,1650.0,NaN,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Dips,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3224,R/C Hollywood Cinema 4,75,866,R/C Theatres,None,amc_github,2024-02-15 22:19:57.087795+00:00,Salty Snacks,75,Pretzel Bites: Add Salt,...,300.0,NaN,NaN,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,Salty Snacks,false
3225,EMC Leominster 10,66,159,Entertainment Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Chips and Dips,66,Tostitos Cheese Sauce,...,620.0,NaN,15.0,1.0,1.0,1.0,amc,2024-02-07 03:53:10.283111+00:00,Chips and Dips,false
3226,EMC Lebanon 6,66,158,Entertainment Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Chips and Dips,66,Tostitos Cheese Sauce,...,620.0,NaN,15.0,1.0,1.0,1.0,amc,2024-02-07 03:53:10.283111+00:00,Chips and Dips,false
3227,Entertainment Cinemas South Dennis,66,476,Entertainment Cinemas,None,amc_github,2024-02-15 22:19:57.087795+00:00,Chips and Dips,66,Tostitos Cheese Sauce,...,620.0,NaN,15.0,1.0,1.0,1.0,amc,2024-02-07 03:53:10.283111+00:00,Chips and Dips,false


In [42]:
%%bigquery
declare prompt_query STRING default "Detect the service category of a snack being offered at a theater. The snacks bought from counter, such as basic snacks or small like dips, candy, chips, popcorn, beverages, drinks, coffee, sauce; these snacks should be grouped into the same snack service. The snacks that must be delivered to the seat such as snacks that take time to make, such as pizza, seafood, sandwiches, baked goods, burgers, etc should be grouped in the same snack_service. Also, if the snack has very high sodium or very high protein or very high calories, they should also be grouped with the snacks that must be delivered to the seat. Return the output as json, include only the theater_id, snack_id, service_category in the output.";
create or replace table movie_stg_ai.snack_service_category_predictions_raw as
select *
from ML.generate_text(
  model remote_models.gemini_pro,
  (
    select concat(prompt_query, to_json_string(json_object("snack_id", ss.snack_id, "theater_id", ss.theater_id, 
                                                           "calories", s.calories, "protein", s.protein, "sodium", s.sodium, "category", s.category)))
      as prompt
    from movie_stg.Snack_Service ss
    LEFT JOIN movie_stg.Snack s ON  ss.snack_id = s.id
    order by theater_id
  ),
  struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
);

Query is running:   0%|          |

""


In [43]:
%%bigquery
select ml_generate_text_llm_result, prompt
from movie_stg_ai.snack_service_category_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,prompt
0,"{""theater_id"":178,""snack_id"":87,""service_categ...",Detect the service category of a snack being o...
1,"{""theater_id"":331,""snack_id"":87,""service_categ...",Detect the service category of a snack being o...
2,"{""theater_id"":368,""snack_id"":87,""service_categ...",Detect the service category of a snack being o...
3,"{""theater_id"":379,""snack_id"":87,""service_categ...",Detect the service category of a snack being o...
4,"{""theater_id"":446,""snack_id"":87,""service_categ...",Detect the service category of a snack being o...
...,...,...
3224,None,Detect the service category of a snack being o...
3225,None,Detect the service category of a snack being o...
3226,None,Detect the service category of a snack being o...
3227,None,Detect the service category of a snack being o...


In [44]:
%%bigquery
create or replace table movie_stg_ai.snack_service_category_predictions_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.snack_service_category_predictions_raw

Query is running:   0%|          |

""


In [45]:
%%bigquery
create or replace table movie_stg_ai.snack_service_category_predictions_formatted as
  select ml_generate_text_llm_result
  from movie_stg_ai.snack_service_category_predictions_raw

Query is running:   0%|          |

""


In [46]:
%%bigquery
select * from movie_stg_ai.snack_service_category_predictions_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,None
1,None
2,None
3,None
4,None
...,...
3224,"{""theater_id"": 6051, ""snack_id"": 51, ""service_..."
3225,"{""theater_id"": 6079, ""snack_id"": 52, ""service_..."
3226,"{""theater_id"": 6097, ""snack_id"": 30, ""service_..."
3227,"{""theater_id"": 6098, ""snack_id"": 31, ""service_..."


In [47]:
%%bigquery
update movie_stg.Snack_Service set service_category =
  (select json_value(ml_generate_text_llm_result, '$.service_category')
   from movie_stg_ai.snack_service_category_predictions_formatted
   where CAST(theater_id AS STRING) = json_value(ml_generate_text_llm_result, '$.theater_id'))
where 1=1;

Query is running:   0%|          |

""


In [48]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack_Service
where service_category is not null

Query is running:   0%|          |

Downloading:   0%|          |

,theater_name,snack_id,theater_id,theater_chain_name,service_category,category
0,B&B Muskogee Arrowhead Mall 10,60,1549,B&B Theatres,Counter,Sauce
1,B&B Iola Sterling Six Cinemas,60,3460,B&B Theatres,Counter,Sauce
2,B&B Moberly 5,60,1729,B&B Theatres,Counter,Sauce
3,B&B Neosho Cinema 6,60,3397,B&B Theatres,Counter,Sauce
4,B&B Overland Park 16,60,3929,B&B Theatres,Counter,Sauce
...,...,...,...,...,...,...
3200,CC Lake Theatre,98,792,Classic Cinemas,delivered_to_seat,Pastries
3201,CC Fox Lake Theatre,98,791,Classic Cinemas,delivered_to_seat,Pastries
3202,CC Ogden 6 Theatre,98,795,Classic Cinemas,delivered_to_seat,Pastries
3203,CC Tivoli Theatre,98,797,Classic Cinemas,delivered_to_seat,Pastries


In [57]:
%%bigquery
update movie_stg.Snack_Service
  set data_source = 'amc_github_ai' where service_category is not null

Query is running:   0%|          |

""


In [58]:
%%bigquery
select *
from movie_stg.Snack_Service

Query is running:   0%|          |

Downloading:   0%|          |

,theater_name,snack_id,theater_id,theater_chain_name,service_category,data_source,load_time
0,The Art Theater,17,2910,Independent Chains,None,amc_github,2024-02-15 22:19:57.087795+00:00
1,The Historic Avalon Theatre,17,5562,Independent Chains,None,amc_github,2024-02-15 22:19:57.087795+00:00
2,Mos' Art Theatre,17,4751,Independent Chains,None,amc_github,2024-02-15 22:19:57.087795+00:00
3,The Corazon Cinema and Cafe,17,5567,Independent Chains,None,amc_github,2024-02-15 22:19:57.087795+00:00
4,Orpheum Theatre - Saugerties,17,2285,Independent Chains,None,amc_github,2024-02-15 22:19:57.087795+00:00
...,...,...,...,...,...,...,...
3224,CC Tivoli Theatre,98,797,Classic Cinemas,delivered_to_seat,amc_github_ai,2024-02-15 22:19:57.087795+00:00
3225,CC North Riverside Mall Theatre,98,4600,Classic Cinemas,delivered_to_seat,amc_github_ai,2024-02-15 22:19:57.087795+00:00
3226,CC Woodstock Theatre,98,798,Classic Cinemas,delivered_to_seat,amc_github_ai,2024-02-15 22:19:57.087795+00:00
3227,CC Paramount,98,796,Classic Cinemas,delivered_to_seat,amc_github_ai,2024-02-15 22:19:57.087795+00:00


In [55]:
# %%bigquery

# CREATE OR REPLACE TABLE movie_stg.Snack_Service AS
# SELECT
#  * EXCEPT(category)
# FROM
#   movie_stg.Snack_Service

Query is running:   0%|          |

""


# Part 4: Merge changes into target table

In [85]:
%%bigquery
alter table movie_csp.Snack
  add column category string;

Query is running:   0%|          |

""


In [86]:
%%bigquery
alter table movie_csp.Snack
  add column is_healthy string;

Query is running:   0%|          |

""


In [87]:
%%bigquery
select count(*) as num_records
from movie_csp.Snack

Query is running:   0%|          |

Downloading:   0%|          |

,num_records
0,104


In [88]:
%%bigquery
select count(*) as num_updates
from movie_csp.Snack t join movie_stg.Snack s
on t.id = s.id
where t.status_flag = true
and (s.category is not null and t.category is null
or s.is_healthy is not null and t.is_healthy is null);

Query is running:   0%|          |

Downloading:   0%|          |

,num_updates
0,100


In [89]:
%%bigquery
declare current_ts TIMESTAMP;
set current_ts = current_timestamp();

create temp table updates as
  select s.*
  from movie_csp.Snack t join movie_stg.Snack s
  on t.id = s.id
  where t.status_flag = true
  and (s.name is not null and t.name is null
  or s.category is not null and t.category is null
  or s.is_healthy is not null and t.is_healthy is null);

update movie_csp.Snack
set discontinue_time = timestamp_sub(current_ts, interval 1 second), status_flag = false
where id in (select id from updates);

insert into movie_csp.Snack
  (id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium,carbohydrates, dietary_fiber, sugars, protein,
    category, is_healthy, data_source, load_time, effective_time, status_flag)
(select id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium,carbohydrates, dietary_fiber, sugars, protein,
    category, is_healthy, data_source, load_time, current_ts, true
      from updates);

Query is running:   0%|          |

""


In [90]:
%%bigquery
declare current_ts TIMESTAMP;
set current_ts = current_timestamp();

create temp table updates as
  select s.*
  from movie_csp.Snack_Service t join movie_stg.Snack_Service s
  on t.theater_id = s.theater_id
  where t.status_flag = true
  and (s.service_category is not null and t.service_category is null);

update movie_csp.Snack_Service
set discontinue_time = timestamp_sub(current_ts, interval 1 second), status_flag = false
where theater_id in (select theater_id from updates);

insert into movie_csp.Snack_Service
  (theater_name, snack_id, theater_id, theater_chain_name,
    service_category, data_source, load_time, effective_time, status_flag)
(select theater_name, snack_id, theater_id, theater_chain_name,
    service_category, data_source, load_time, current_ts, true
      from updates);

Query is running:   0%|          |

""


In [75]:
%%bigquery
select distinct effective_time, discontinue_time, status_flag
from movie_csp.Snack

Query is running:   0%|          |

Downloading:   0%|          |

,effective_time,discontinue_time,status_flag
0,2024-02-20 05:08:39.286737+00:00,2024-04-10 23:21:29.084110+00:00,False
1,2024-03-31 00:12:53.438792+00:00,2024-04-10 23:21:29.084110+00:00,False
2,2024-03-31 00:13:13.682948+00:00,2024-04-10 23:21:29.084110+00:00,False
3,2024-02-20 05:08:39.286737+00:00,2024-03-31 00:12:53.438792+00:00,False


In [77]:
%%bigquery
select distinct effective_time, discontinue_time, status_flag
from movie_csp.Snack_Service

Query is running:   0%|          |

Downloading:   0%|          |

,effective_time,discontinue_time,status_flag
0,2024-02-20 05:11:04.542698+00:00,2024-04-10 23:27:06.402370+00:00,False
1,2024-02-20 05:11:04.542698+00:00,NaT,True
2,2024-04-10 23:27:07.402370+00:00,NaT,True


In [93]:
%%bigquery
select * from movie_csp.Snack
where id = 81

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,...,dietary_fiber,sugars,protein,data_source,load_time,effective_time,discontinue_time,status_flag,category,is_healthy
0,81,Mango Smoothie 16 fl oz,270.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,68.0,NaN,amc_ai,2024-02-07 03:53:10.283111+00:00,2024-04-11 00:14:47.011336+00:00,NaT,True,Smoothies,false
1,81,Mango Smoothie 16 fl oz,270.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,68.0,NaN,amc,2024-02-07 03:53:10.283111+00:00,2024-02-20 05:08:39.286737+00:00,2024-04-11 00:14:46.011336+00:00,False,None,None


In [96]:
%%bigquery
select * from movie_csp.Snack_Service
where theater_id = 2981

Query is running:   0%|          |

Downloading:   0%|          |

,theater_name,snack_id,theater_id,theater_chain_name,service_category,data_source,load_time,effective_time,discontinue_time,status_flag
0,Marcus ULTRACINEMA Oakdale,52,2981,Marcus Theaters,None,amc_github,2024-02-15 22:19:57.087795+00:00,2024-02-20 05:11:04.542698+00:00,2024-04-10 23:27:06.402370+00:00,False
1,Marcus ULTRACINEMA Oakdale,52,2981,Marcus Theaters,delivered_to_seat,amc_github_ai,2024-02-15 22:19:57.087795+00:00,2024-04-10 23:27:07.402370+00:00,NaT,True
